In [ ]:
import urllib
import time, timeit
from math import sqrt

# Introducción
### Primer ejemplo: range

Los generadores son una funcionalidad de Python que sirven para no generar de golpe todos los elementos sobre los que vamos a iterar en un bucle *for*. Cuando utilizamos los generadores, los elementos se van generando a demanda. 

Para empezar, veamos un ejemplo sencillo. Para iterar sobre una lista de números consecutivos, lo hacemos a través de *range*. Sin embargo, también podemos generar una lista e iterar sobre ella.

In [ ]:
def crear_iterable(max_val, opción):
    if opción == 1:
        return [i for i in range(max_val)] # Lista
    elif opción == 2:
        return range(max_val) # Generador

Si le pasamos como argumento "1", nos devuelve una lista con todos los elementos cargados.
Si le pasamos como argumento "2", nos devuelve el range habitual, que es un generador.

Si queremos obtener la suma de todos los elementos, ¿qué será más eficiente?

In [ ]:
max_val = 10_000_000

In [ ]:
%%timeit # Esta función repite varias veces la celda para ver cuánto tarda en ejecutarse
suma = 0
for i in crear_iterable(max_val, 1):
    suma += i

In [ ]:
%%timeit 
suma = 0
for i in crear_iterable(max_val, 2):
    suma += i

La opción del generador es bastante más rápida. Esto es porque la cantidad de memoria que tenemos que cargar es bastante menor en el segundo caso, ya que solo hay un número de la iteración cargado a la vez.

Sin embargo, ¿qué pasa si queremos parar a mitad del bucle?

In [ ]:
%%timeit # Esta función repite varias veces la celda para ver cuánto tarda en ejecutarse
suma = 0
for i in crear_iterable(max_val, 1):
    if i == 100:
        break

In [ ]:
%%timeit # Esta función repite varias veces la celda para ver cuánto tarda en ejecutarse
suma = 0
for i in crear_iterable(max_val, 2):
    if i == 100:
        break

¡La diferencia es abismal! No tenemos que generar toda la lista de números, sino que con un generador solo se van a crear los 100 primeros números.

### Segundo ejemplo: ficheros

A la hora de tratar con ficheros ocurre lo mismo. Podemos usar un generador para leer línea a línea o listar de golpe todas las líneas con un *readlines()*.

Vamos a descargarnos El Quijote para comprobar este comportamiento. Supongamos que queremos imprimir toda la información antes del primer capítulo.

In [ ]:
# Descargamos El Quijote
url = "https://babel.upm.es/~angel/teaching/pps/quijote.txt"
archivo_local = "quijote.txt"

try:
    urllib.request.urlretrieve(url, archivo_local)
    print(f"Archivo descargado exitosamente como '{archivo_local}'.")
except Exception as e:
    print(f"Error al descargar el archivo: {e}")

In [ ]:
%%timeit
# Opción A: usar la lista
file = open("quijote.txt", "r", encoding="latin-1")
for line in file.readlines():
    if "Capítulo primero" in line:
        break
    # Aquí podríamos hacer un print, procesar el texto, etc.

file.close()

In [ ]:
%%timeit
# Opción B: usar generadores
file = open("quijote.txt", "r", encoding="latin-1")
for line in file:
    if "Capítulo primero" in line:
        break
    # Aquí podríamos hacer un print, procesar el texto, etc.
file.close()

La diferencia se nota menos que antes porque la función *open* ocupa una fracción del tiempo en ambos casos. Sin embargo, con textos mayores la diferencia se irá notando más.

# Creación de generadores

Además de los generadores que vienen por defecto en Python, podemos crear los nuestros propios. Para ello, se utiliza la sentencia *yield* en vez de *return*. Veamos las diferencias teóricas:
- Cuando utilizamos *return*, la función devuelve un cierto valor y __su ejecución se detiene__. En el caso que nos aplica, devolveríamos una lista.
- Cuando utilizamos *yield*, devolvemos un cierto valor y __la ejecución de la función se pausa__. Si se necesitan más valores (es decir, si cambiamos de iteración), la ejecución de la función se reanuda.

Veamos esto con un ejemplo muy útil: la generación de números primos. Vamos a crear una lista normal y luego un generador de números primos. 

In [ ]:
def lista_primos(maximo):
    # Vamos guardando los primos, ya que son la lista de posibles divisores 
    primos = [] 
    for n in range(2, maximo):
        for p in primos:
            # Hemos acabado la lista de primos que pueden ser divisores
            if p > sqrt(n):
                primos.append(n)
            # Hemos encontrado un divisor
            if n % p == 0:
                break
    return primos


In [ ]:
def generador_primos(maximo):
    for n in range(2, maximo):
        es_primo = True
        for i in range(2, int(sqrt(n))):
            # Hemos encontrado un divisor
            if n % i == 0:
                es_primo = False
                break
        if es_primo:
            yield n


Vamos a obtener la lista de los primeros 100 números primos. No sabemos cuál es el número máximo que necesitaremos, así que ponemos un millón.

In [ ]:
%%timeit
lst = list()
cnt = 0
for primo in lista_primos(1_000_000):
    cnt+=1
    lst.append(primo)
    if cnt == 100:
        break

In [ ]:
%%timeit
lst = list()
cnt = 0
for primo in generador_primos(1_000_000):
    cnt+=1
    lst.append(primo)
    if cnt == 100:
        break

El generador es muchísimo más rápido, incluso si tiene que comprobar todos los números en vez de la lista de primos.

Sin embargo, los generadores nos dan una ventaja competitiva más... ¡podemos generar valores infinitos! Sin embargo, las listas sí deben tener una longitud determinada. Esto significa que nos podemos olvidar de determinar un número máximo y podemos crear un generador infinito de primos.

In [ ]:
def generador_primos():
    n = 2
    while True:
        es_primo = True
        for i in range(2, int(sqrt(n)) + 1):
            # Hemos encontrado un divisor
            if n % i == 0:
                es_primo = False
                break
        if es_primo:
            yield n
        n += 1

In [ ]:
num = 2_000

for i, n in enumerate(generador_primos()):
    if i == num:
        print(f"El número primo número {num} es {n}")
        break

## Comprensión de listas

En Python, el ahorro de líneas de código es una característica muy valorada. Una de las prácticas más habituales a la hora de programar consiste en crear una lista a partir de un bucle *for*, definiendo cada elemento en una iteración distinta.

Por ejemplo, vamos a crear una lista de las primeras raíces cuadradas que provengan de números pares:

In [ ]:
# longitud de la lista
list_len = 10_000_000

In [ ]:
%%timeit
lista_raices = []

for i in range(list_len):
    if i % 2 == 0:
        lista_raices.append(sqrt(i))

Sin embargo, en Python existe la comprensión de listas (list comprehension), que permite definir los elementos de una lista según se define la lista. Para este ejemplo, la solución sería:

In [ ]:
%%timeit
lista_raices = [sqrt(i) for i in range(list_len) if i % 2 == 0]

Si este código es algo más rápido, es porque Python sabe de primera mano cuántos elementos vamos a introducir en nuestra lista, en vez de reservar constantemente memoria para un nuevo elemento en la lista sin saber cuántos habrá.

La comprensión de listas también sirve para definir generadores, que utilizan paréntesis en vez de corchetes:

In [ ]:
%%timeit
generador_raices = (sqrt(i) for i in range(list_len) if i % 2 == 0)

De hecho, también sirve para diccionarios, pero debemos definir clave y valor.

In [ ]:
dict_raices = {i:sqrt(i) for i in range(list_len) if i % 2 == 0}

Como último ejemplo, se muestra una lista más complicada, con varios bucles for.

In [ ]:

# ¿Qué debería devolver?
pares_filtrados = [(x, y) 
                   for x in range(1, 6) 
                   for y in range(1, 6) 
                   if x < y and (x + y) % 2 == 0]
